# BLIP image-captioning-base — DIMER image captioning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/blip-captioning-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/blip-captioning-pipeline/blob/main/tutorials/blip_captioning_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Salesforce%2Fblip--image--captioning--base-ffcc4d?style=flat)](https://huggingface.co/Salesforce/blip-image-captioning-base) [![Upstream](https://img.shields.io/badge/Upstream-salesforce%2FBLIP-181717?style=flat&logo=github&logoColor=white)](https://github.com/salesforce/BLIP) [![arXiv](https://img.shields.io/badge/arXiv-2201.12086-b31b1b.svg)](https://arxiv.org/abs/2201.12086)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** Image captioning — one image (optionally with a caption prefix to continue) → one sentence of generated text — using the pinned `Salesforce/blip-image-captioning-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/blip_captioning_pipeline/pipeline.py` at revision `9cf80dc266c2`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `82a37760796d32b1411fe092ab5d4e227313294b` (~991 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the BLIP model (a ViT-B/16 image encoder at 384×384 and a 12-layer BERT-style text decoder that cross-attends to the image features; about 247M parameters, pretrained on 129M image–text pairs with captioning-and-filtering bootstrapping and fine-tuned on COCO Captions) encodes the resized image and generates a caption token by token, either from scratch (unconditional) or continuing a prefix you supply such as `a photography of` (conditional, the upstream README's example). Decoding is greedy (`do_sample=False`, one beam) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, an optional prefix up to 128 characters, the token budget), a fixed output contract, and the `keyword_hits`, `unigram_f1`, `validate_inputs` and `evaluation_report` helpers. **Weight-format note:** upstream hosts no SafeTensors at the pinned revision; the carried module executes the digest-pinned `pytorch_model.bin` (a pickle, deserialised with `weights_only=True` after its SHA-256 is checked), while the `tf_model.h5` upstream also hosts is DIMER's upload artifact and is never loaded here. The default sample is three flat cartoon scenes drawn in code with no reference captions, so the evaluation report is `not-measurable` by design and the printed keyword checks are observations, not a captioning benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision (a pickle checkpoint, and why that matters), draw three synthetic scenes (or upload your own photographs) and validate them into an input manifest, choose a token budget and an optional prefix, run the supported task, read the captions correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `not-measurable` without reference captions and `sample-sanity` with a bag-of-words `unigram_f1` when you supply some, and export the captions, an annotated contact sheet and provenance.

**This notebook does not demonstrate:** Reading text in the image (BLIP is not an OCR model), visual question answering (a separate checkpoint), dense or region captioning (one sentence per image, no localisation), captions in languages other than English, batch throughput, sampling, beam search or repetition penalties (the upstream evaluation used beam search; this notebook decodes greedily for reproducibility), evaluation on COCO Captions (not bundled; CIDEr, BLEU-4 and SPICE need several references per image and are not computed here), and any training. The model was fine-tuned on photographs; drawings, diagrams, documents and expert imagery are outside what this notebook measures, and a fluent wrong caption carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 3.9 s to load and 0.2–0.5 s per caption on 640×480 drawn scenes in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 990 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; why a pickle checkpoint needs a digest check before `torch.load`; what reference-based caption metrics (CIDEr, BLEU) need; that a confident caption is not a correct one.
- **Data:** the default sample is three deterministic cartoon scenes drawn in code with Pillow (a house with a tree and the sun; a beach with a sailboat; two fruits on a table — no text rendering, so their digests are stable across Pillow builds) with **no reference captions**, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one or more images decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Salesforce/blip-image-captioning-base` snapshot (~991 MB in total) at revision `82a37760796d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'blip-captioning-pipeline',
    'repository_revision': '9cf80dc266c21824196d61d2198cb8d40032ac9a',
    'embedded_module': 'src/blip_captioning_pipeline/pipeline.py',
    'embedded_modules': ['src/blip_captioning_pipeline/pipeline.py'],
    'module_sha256': '317447a72a0983164b61243436af7f0af11955f5481b8f4c49a404b5de7e5a4c',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/blip_captioning_pipeline/` @ `9cf80dc266c2`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/blip_captioning_pipeline/pipeline.py`

In [ ]:
"""Image captioning with the pinned ``Salesforce/blip-image-captioning-base`` checkpoint (BLIP, ViT-B).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the BLIP architecture comes from the pinned ``transformers`` release and no
model-repository code is executed. Upstream ships no SafeTensors at this revision: the PyTorch weights
are ``pytorch_model.bin`` (a pickle), so the trust boundary is the manifest SHA-256 checked before the
load plus ``weights_only=True`` deserialisation; the ``tf_model.h5`` upstream also hosts is the DIMER
upload artifact and is never loaded here.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "Salesforce/blip-image-captioning-base"
MODEL_REVISION = "82a37760796d32b1411fe092ab5d4e227313294b"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "blip-image-captioning-base"
WEIGHT_FILE = "pytorch_model.bin"  # the only PyTorch weight file upstream: a pickle, digest-pinned
HOSTED_TF_WEIGHT_FILE = "tf_model.h5"  # DIMER upload artifact (accepted format); never loaded by this package
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. COCO-style captions are one sentence (the checkpoint's text_config max_length
# is 20); the default leaves room for a long sentence and the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 30
DECODING = "greedy"
# Optional conditional-captioning prefix (the upstream README's "a photography of"); the model
# continues it. A prefix longer than a short phrase is not what the model was trained on.
MAX_PREFIX_CHARS = 128
# Input ceilings. The processor resizes every image to 384x384 (preprocessor_config.json, aspect
# ratio not preserved) into 24x24 = 576 ViT-B/16 patches, so image cost is bounded; the side ceiling
# only guards memory during decoding and resizing.
IMAGE_SIZE = 384
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_caption(text: str) -> str:
    """COCO-caption-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def caption_tokens(text: str) -> list[str]:
    return normalize_caption(text).split()


def unigram_f1(prediction: str, references: Sequence[str]) -> float:
    """Bag-of-words F1 between the normalised prediction and the best-matching reference.

    A plumbing check, not a captioning metric: CIDEr, BLEU-4 and SPICE need several references per
    image and corpus-level statistics. Multiset overlap counts repeated words once per occurrence.
    """
    if not references:
        raise ValueError("references must contain at least one caption")
    pred = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        if not pred or not ref:
            continue
        ref_counts: dict[str, int] = {}
        for token in ref:
            ref_counts[token] = ref_counts.get(token, 0) + 1
        overlap = 0
        for token in pred:
            if ref_counts.get(token, 0) > 0:
                overlap += 1
                ref_counts[token] -= 1
        if overlap:
            precision, recall = overlap / len(pred), overlap / len(ref)
            best = max(best, 2 * precision * recall / (precision + recall))
    return best


def keyword_hits(caption: str, keywords: Sequence[str]) -> dict[str, bool]:
    """Which of the caller's keywords (normalised, whole-token match) appear in the caption."""
    tokens = set(caption_tokens(caption))
    return {keyword: all(part in tokens for part in caption_tokens(keyword)) for keyword in keywords}


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB) plus an optional caption prefix",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prefix_chars": [0, MAX_PREFIX_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False, num_beams=1), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image resized to 384x384 (aspect ratio not preserved, CLIP mean/std) into 576 ViT-B/16 patches; "
        "the text decoder starts from the BOS token (unconditional) or from the tokenised prefix "
        "(conditional) and generates the caption"
    ),
    "output": "one caption string (the model's decoded text, prefix included when given), no score",
}


def _check_inputs(image: Any, prefix: Any, max_new_tokens: Any) -> tuple[Image.Image, str | None, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``caption`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    checked_prefix: str | None = None
    if prefix is not None:
        if not isinstance(prefix, str):
            raise TypeError("prefix must be a str or None")
        checked_prefix = " ".join(prefix.split())
        if not checked_prefix:
            raise ValueError("prefix must contain at least one non-whitespace character or be None")
        if len(checked_prefix) > MAX_PREFIX_CHARS:
            raise ValueError(f"prefix has {len(checked_prefix)} chars > MAX_PREFIX_CHARS {MAX_PREFIX_CHARS}")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_prefix, max_new_tokens


def validate_inputs(
    images: Sequence[Image.Image],
    *,
    prefix: str | None = None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every image is checked exactly as ``caption`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(images, Image.Image) or not isinstance(images, Sequence) or not images:
        raise TypeError("images must be a non-empty sequence of PIL.Image.Image")
    if names is not None and len(names) != len(images):
        raise ValueError(f"names has {len(names)} entries for {len(images)} images")
    checked_prefix = None
    observed = []
    for index, image in enumerate(images):
        _, checked_prefix, _ = _check_inputs(image, prefix, max_new_tokens)
        observed.append(
            {"id": names[index] if names else f"image-{index}", "mode": image.mode, "size": list(image.size)}
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "prefix": checked_prefix,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    references: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``references`` (one sequence of reference captions per result, in order) the report carries
    the mean ``unigram_f1`` over the images plus one per-image entry, verdict ``sample-sanity``;
    without references it is ``not-measurable`` and says what labelled data would make the task
    measurable. Neither is a captioning benchmark.
    """
    if not results:
        raise ValueError("results must contain at least one caption result")
    base = {
        "task": "image (+ optional prefix) -> caption text (image captioning)",
        "score_semantics": (
            "the caption is generated text and carries no score, probability or correctness signal; a "
            "fluent caption is not evidence that it describes the image. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_images": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if references is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference captions were supplied for the captioned images",
            "needs": (
                "several human-written reference captions per image from the deployment domain "
                "(COCO Captions-style annotations, five per image) scored with CIDEr / BLEU-4 / SPICE "
                "over a corpus; no such labelled set ships with this repository"
            ),
        }
    if len(references) != len(results):
        raise ValueError(f"references has {len(references)} entries for {len(results)} results")
    per_image = []
    for result, refs in zip(results, references, strict=True):
        if isinstance(refs, str) or not refs:
            raise ValueError("each references entry must be a non-empty sequence of captions")
        prediction = str(result["caption"])
        per_image.append(
            {
                "image": result.get("image"),
                "prediction": prediction,
                "references": list(refs),
                "unigram_f1": unigram_f1(prediction, refs),
            }
        )
    metrics = [
        {
            "id": "unigram_f1",
            "value": sum(entry["unigram_f1"] for entry in per_image) / len(per_image),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; best reference",
            "relation_to_benchmarks": (
                "bag-of-words overlap with the best reference; not CIDEr, BLEU-4 or SPICE, which need "
                "several references per image and corpus-level statistics"
            ),
            "estimation": f"{len(per_image)} image(s), no dispersion estimate",
        }
    ]
    return {
        **base,
        "metrics": metrics,
        "per_image": per_image,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_image)} image(s) with caller-written reference captions; plumbing evidence, not a "
            "captioning benchmark"
        ),
        "needs": (
            "several human-written reference captions per image from the deployment domain scored with "
            "CIDEr / BLEU-4 / SPICE over a corpus for any quality claim; COCO Captions is not bundled"
        ),
    }


@dataclass
class BlipCaptioningPipeline:
    """``_runner(image, prefix, max_new_tokens)`` returns ``{"caption": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BlipCaptioningPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BlipForConditionalGeneration, BlipProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = BlipProcessor.from_pretrained(location, **common)
        # Trust boundary: the only PyTorch weight file upstream is a pickle (pytorch_model.bin). Its
        # SHA-256 was checked against the manifest above; use_safetensors=False names that fact, and
        # weights_only=True makes transformers deserialise with torch.load(weights_only=True), whose
        # restricted unpickler admits tensors, primitives and containers only.
        model = BlipForConditionalGeneration.from_pretrained(
            location, dtype=torch.float32, use_safetensors=False, weights_only=True, **common
        )
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, prefix: str | None, max_new_tokens: int) -> dict[str, Any]:
            if prefix is None:
                inputs = processor(images=image, return_tensors="pt").to(resolved_device)
                prompt_len = 0
            else:
                inputs = processor(images=image, text=prefix, return_tensors="pt").to(resolved_device)
                prompt_len = int(inputs["input_ids"].shape[1])
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            ids = generated[0]
            decoded = processor.decode(ids, skip_special_tokens=True)
            # Unconditional: bos + caption + sep. Conditional: the prompt ids minus their trailing
            # [SEP] are echoed first (BlipForConditionalGeneration.generate drops it), then the new ones.
            new_tokens = int(ids.shape[0]) - (prompt_len - 1 if prompt_len else 1)
            return {"caption": decoded, "new_tokens": max(new_tokens, 0)}

        return cls(runner, resolved_device, "float32", source)

    def caption(
        self,
        image: Image.Image,
        *,
        prefix: str | None = None,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Caption one image; ``caption`` is the decoded text, stripped (prefix included when given)."""
        rgb, checked_prefix, checked_tokens = _check_inputs(image, prefix, max_new_tokens)
        raw = self._runner(rgb, checked_prefix, checked_tokens)
        if not isinstance(raw, dict) or "caption" not in raw:
            raise RuntimeError("runner must return a dict with 'caption'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "caption": str(raw["caption"]).strip(),
            "prefix": checked_prefix,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `82a37760796d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BlipCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "blip-image-captioning-base",
  "modelId": "Salesforce/blip-image-captioning-base",
  "revision": "82a37760796d32b1411fe092ab5d4e227313294b",
  "files": [
    {
      "path": "README.md",
      "bytes": 6359,
      "sha256": "eae8c14a066e611a51a803cccb35c892bbd78d1ca7db45fbb06f5fb79bff7ad1"
    },
    {
      "path": "config.json",
      "bytes": 4563,
      "sha256": "f7846f82f4e2c4a2ccbab9ae8b0e44873540a271a76a1288effd078180c13a82"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 287,
      "sha256": "065c10ec97edc5081fbd9655b3d9d25e2647ea6d5dab873325e88eed12d80a7d"
    },
    {
      "path": "pytorch_model.bin",
      "bytes": 989820849,
      "sha256": "d6638651a5526cc2ede56f2b5104d6851b0755816d220e5e046870430180c767"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 506,
      "sha256": "5c7f96096284c55e539eff95f4451c19efc34258ddb975a4400d052b77301e5e"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 990775593
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BlipCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scenes or optional BYOD

The default sample is **synthetic**: three flat cartoon scenes — a red house with a brown roof and door, a tree, a white ball and the sun on grass under a blue sky; a beach with sea, sand, a red sailboat and the sun; a red apple and an orange on a wooden table — are drawn with Pillow (640×480, 640×480, 480×480), the same drawings the repository's smoke run used. **No reference captions are authored**, because a caption you write yourself is not an annotation standard: the evaluation report will therefore be `not-measurable`, and the notebook instead records which of a few expected keywords (`house`, `tree`, `beach`, `apple`, …) appear in each caption as an observation. The image digests are printed for the record. BYOD is optional and disabled by default; when enabled, upload one or more images.

Two **caller-owned request parameters** are exposed: `max_new_tokens` bounds the caption (`DEFAULT_MAX_NEW_TOKENS = 30` fits any COCO-style sentence; `MAX_NEW_TOKENS = 64` is the ceiling), and `caption_prefix` (empty for unconditional captioning; `a photography of` is the upstream README's example — the model continues whatever you start, so the prefix shapes the caption). Nothing is validated in this cell — the next section hands the images to the pipeline's own validation stage, which is the only checker. Look for one dictionary per image naming the sample kind, size and digest, plus the budget and prefix.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
max_new_tokens = 30  # @param {type:"integer"}
caption_prefix = ''  # @param {type:"string"}


def synthetic_scenes():
    """Three flat cartoon scenes drawn with Pillow (no text); returns [(name, image, expected keywords)]."""
    house = Image.new('RGB', (640, 480), (135, 206, 235))  # sky
    d = ImageDraw.Draw(house)
    d.rectangle([0, 300, 640, 480], fill=(60, 179, 75))  # grass
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([120, 180, 320, 330], fill=(200, 40, 40))  # red house
    d.polygon([(100, 180), (220, 90), (340, 180)], fill=(90, 50, 20))  # brown roof
    d.rectangle([200, 260, 240, 330], fill=(70, 40, 20))  # brown door
    d.ellipse([420, 260, 520, 360], fill=(40, 100, 40))  # tree crown
    d.rectangle([460, 350, 480, 420], fill=(90, 60, 30))  # trunk
    d.ellipse([60, 380, 140, 440], fill=(255, 255, 255))  # white ball
    beach = Image.new('RGB', (640, 480), (120, 190, 240))  # sky
    d = ImageDraw.Draw(beach)
    d.rectangle([0, 220, 640, 330], fill=(30, 110, 200))  # sea
    d.rectangle([0, 330, 640, 480], fill=(238, 214, 150))  # sand
    d.ellipse([60, 40, 150, 130], fill=(255, 230, 80))  # sun
    d.polygon([(400, 330), (470, 330), (435, 210)], fill=(230, 40, 40))  # red sail
    d.rectangle([432, 210, 438, 330], fill=(90, 60, 30))  # mast
    d.ellipse([200, 370, 260, 430], fill=(255, 120, 40))  # beach ball
    fruit = Image.new('RGB', (480, 480), (250, 250, 245))
    d = ImageDraw.Draw(fruit)
    d.ellipse([60, 120, 220, 280], fill=(220, 30, 30))  # red apple
    d.rectangle([135, 95, 145, 125], fill=(80, 50, 20))  # stalk
    d.ellipse([250, 140, 430, 300], fill=(255, 170, 20))  # orange
    d.polygon([(90, 400), (400, 400), (360, 330), (130, 330)], fill=(180, 120, 60))  # table
    return [
        ('synthetic_house_640x480.png', house, ['house', 'tree', 'red']),
        ('synthetic_beach_640x480.png', beach, ['beach', 'sail', 'sun']),
        ('synthetic_fruit_480x480.png', fruit, ['apple', 'orange']),
    ]


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    samples = []
    for name, data in uploaded.items():
        image = Image.open(io.BytesIO(data))
        image.load()
        samples.append((name, image, []))
    sample_kind = 'BYOD'
else:
    # Deterministic drawings: no randomness and no text rendering, so no seed is needed and the digests are stable.
    samples = synthetic_scenes()
    sample_kind = 'synthetic'

names = [name for name, _, _ in samples]
images = [image for _, image, _ in samples]
expected_keywords = [keywords for _, _, keywords in samples]
prefix = caption_prefix.strip() or None
digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in zip(names, images)}
for name, image in zip(names, images):
    print({'sample_kind': sample_kind, 'name': name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': digests[name]})
print({'max_new_tokens': max_new_tokens, 'prefix': prefix, 'n_images': len(images)})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `caption` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, an optional prefix that is a non-empty string of at most `MAX_PREFIX_CHARS` characters (whitespace collapsed), and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the 384×384 resize that does not preserve aspect ratio, and the decoding rule), each input's observed mode and size, the checked prefix, the budget and the verdict. The manifest is written to `outputs/blip_captioning_input_manifest.json`. To show what rejection looks like, the cell also validates a 4-pixel image and records the pipeline's own error message as a finding. Inside the pipeline each image is converted to RGB and resized to `IMAGE_SIZE`×`IMAGE_SIZE`; nothing else is dropped or altered. The pipeline cannot tell whether an image is a photograph or a drawing: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'IMAGE_SIZE': IMAGE_SIZE, 'MAX_PREFIX_CHARS': MAX_PREFIX_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(images, prefix=prefix, max_new_tokens=max_new_tokens, names=names)
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs([Image.new('RGB', (4, 4))])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'tiny-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/blip_captioning_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Caption the images and read the output correctly

`caption` returns, per image, a dict with `caption` (the decoded text, stripped; the prefix is echoed at the start when one was given), the checked `prefix`, `image_size`, `new_tokens` (tokens generated after the prefix, end-of-sequence included), a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the caption is generated text with no probability and no correctness signal, and a fluent caption is not evidence that it describes the image. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the sentence, so GPU and CPU outputs need not match. Each call re-encodes the image, so cost is per image (about 0.2–0.5 s on the reference CPU). As recorded in the model card, the repository's CPU smoke captioned these same drawings `a red house with a tree and a ball`, `a beach scene with a sail and a sun` and `a red and yellow apple on a white background` (the orange became a second apple) — and captioned a blank white image `a white and black striped rug with a white border` and uniform noise `a very colorful and very colorful tv screen`: the model always produces a caption, whether or not there is anything to describe. The cell also records which expected keywords appear in each caption; that is an observation, not a metric.

In [ ]:
import time

results, seconds = [], []
for name, image in zip(names, images):
    t0 = time.time()
    result = pipe.caption(image, prefix=prefix, max_new_tokens=max_new_tokens)
    result['image'] = name
    results.append(result)
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_image': seconds, 'any_truncated': any(r['truncated'] for r in results)})
keyword_observations = []
for result, keywords in zip(results, expected_keywords):
    hits = keyword_hits(result['caption'], keywords) if keywords else {}
    keyword_observations.append({'image': result['image'], 'expected_keywords': keywords, 'hits': hits})
    print(f"{result['image']}\n   caption: {result['caption']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})\n   keywords: {hits}")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that caption is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No quality is reported by default: caption metrics (CIDEr, BLEU-4, SPICE) need several human-written reference captions per image from the deployment domain and corpus-level statistics, and this repository ships none (COCO Captions is not bundled). The repository's helper `unigram_f1` — bag-of-words F1 after normalisation (lower-case, punctuation removed, whitespace collapsed) against the best-matching reference — exists so that a caller who does supply references gets a `sample-sanity` report with one entry per image; it is explicitly **not** a captioning metric. On the default path no references are supplied, the verdict is `not-measurable`, and the report states what would make the task measurable; the keyword observations from the previous section are attached to the report file under `observations` for the record. The report is written to `outputs/blip_captioning_evaluation_report.json`. To see the other branch, set `references` below to one list of reference captions per image.

In [ ]:
references = None  # e.g. [['a red house with a tree'], ['a sailboat on the sea'], ['an apple and an orange on a table']]
report = evaluation_report(results, references, sample_kind=sample_kind)
report['observations'] = keyword_observations
with open('outputs/blip_captioning_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_image', 'observations')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_image', []):
    print(f"  unigram_f1 {entry['unigram_f1']:.2f}  {entry['image']} -> {entry['prediction']!r} (references: {entry['references']})")
if report['verdict'] == 'not-measurable':
    print('No reference captions exist for these images, so nothing is scored; read the captions against the images yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (image, caption, prefix, `new_tokens`, `truncated`, the budget), the evaluation report with the keyword observations, the input manifest, the sample identities and digests, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the executed weight file and the hosted TensorFlow file it is not, and the runtime identity (Python, `torch`, `transformers`, device). The captions are also written as CSV with explicit `image`, `prefix`, `caption`, `new_tokens`, `truncated` columns, and an annotated PNG contact sheet shows each image with its caption printed beneath it for visual inspection (the model returns no location, so nothing is drawn on the images themselves) — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

thumb_w, thumb_h, panel_h = 320, 240, 44
sheet = Image.new('RGB', (thumb_w * len(images), thumb_h + panel_h), 'white')
draw = ImageDraw.Draw(sheet)
panel_font = ImageFont.load_default(size=13)
for index, (image, result) in enumerate(zip(images, results)):
    thumb = image.convert('RGB').copy()
    thumb.thumbnail((thumb_w, thumb_h))
    sheet.paste(thumb, (index * thumb_w + (thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2))
    draw.text((index * thumb_w + 6, thumb_h + 6), result['caption'][:60], fill=(40, 90, 220), font=panel_font)
    if len(result['caption']) > 60:
        draw.text((index * thumb_w + 6, thumb_h + 24), result['caption'][60:120], fill=(40, 90, 220), font=panel_font)
sheet.save('outputs/blip_captioning_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'names': names, 'sizes': [list(image.size) for image in images], 'rgb_sha256': digests, 'expected_keywords': expected_keywords},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'executed_weight_file': WEIGHT_FILE,
    'hosted_tf_weight_file_not_loaded': HOSTED_TF_WEIGHT_FILE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/blip_captioning_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/blip_captioning_captions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'prefix', 'caption', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([result['image'], result['prefix'] or '', result['caption'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The captions are the text the model generates for an image; nothing in the output scores that text, the model returns no location or evidence, and it captions every image — including a blank one — with equal fluency. On the drawn scenes the evaluation report is `not-measurable` by design: no reference captions exist, and the keyword observations (the repository's smoke run found `house`, `tree`, `red`, `beach`, `sail`, `sun` and `apple`, and called the orange a second apple) are what you can check by eye, not a metric; they say nothing about photographs, cluttered scenes, counting, reading, attributes the model tends to hallucinate (colours, backgrounds) or captions longer than a sentence, and a BYOD result is a per-image observation with the same verdict. **The model captions any image** and stops only at end-of-sequence or the token budget: check `truncated`, and treat a plausible caption of an empty or meaningless image as the expected failure mode, not an exception. A prefix steers the caption — `a photography of` produced `a photography of a red house in a green field` for the house scene — so a conditional caption is partly your sentence. The pipeline provides no OCR, no VQA, no region captioning, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model (a pickle checkpoint loaded with `weights_only=True` only after its digest matched), validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `caption_prefix` to `a drawing of` and compare (the smoke run got `a drawing of a house and tree`); lower `max_new_tokens` to 3 and watch `truncated` turn true on `a red house`; write one reference caption per image into `references` and see the verdict switch to `sample-sanity` with a `unigram_f1` you should not mistake for CIDEr; enable `USE_BYOD` with photographs you know.

## References

- Repository README: https://github.com/kurtvalcorza/blip-captioning-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/blip-captioning-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/blip-captioning-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Salesforce/blip-image-captioning-base
- Upstream code: https://github.com/salesforce/BLIP
- BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation (Li et al., 2022): https://arxiv.org/abs/2201.12086
- Microsoft COCO Captions: Data Collection and Evaluation Server (Chen et al., 2015): https://arxiv.org/abs/1504.00325
- CIDEr: Consensus-based Image Description Evaluation (Vedantam, Zitnick, Parikh, 2015): https://arxiv.org/abs/1411.5726